In [3]:
from pathlib import Path
import sys

import certifi
import requests

from azure.core.credentials import AzureNamedKeyCredential
from azure.storage.blob import BlobServiceClient
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


OVERWRITE = True

ARQUIVOS = [
    {
        "nome": "pib_municipal.json",
        "url": (
            "https://apisidra.ibge.gov.br/"
            "values/t/5938/n6/all/"
            "v/37,516,520,528,6574/p/2021"
        ),
    },
    {
        "nome": "censo_2022.json",
        "url": (
            "https://apisidra.ibge.gov.br/"
            "values/t/4714/n6/all/"
            "v/93,6318,614/p/2022"
        ),
    },
]


def detectar_databricks():
    try:
        dbutils.secrets
        return True
    except (NameError, AttributeError):
        return False


def localizar_raiz_projeto():
    atual = Path.cwd().resolve()

    for candidato in [atual, *atual.parents]:
        if (candidato / "src").is_dir():
            return candidato

    raise FileNotFoundError(
        "Pasta 'src' não encontrada no diretório atual "
        "ou nos diretórios pais."
    )


def exigir_valor(valor, nome):
    valor = (
        str(valor).strip()
        if valor is not None
        else ""
    )

    if not valor:
        raise RuntimeError(
            f"{nome} não configurado."
        )

    return valor


IS_DATABRICKS = detectar_databricks()
BASE_DIR = localizar_raiz_projeto()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(BASE_DIR),
    )

if not IS_DATABRICKS:
    from dotenv import load_dotenv

    ENV_PATH = BASE_DIR / ".env"

    if not ENV_PATH.is_file():
        raise FileNotFoundError(
            f"Arquivo .env não encontrado: "
            f"{ENV_PATH}"
        )

    ambiente_carregado = load_dotenv(
        dotenv_path=ENV_PATH,
        override=True,
    )

    if not ambiente_carregado:
        raise RuntimeError(
            f"Não foi possível carregar: "
            f"{ENV_PATH}"
        )

from src.config.secrets import get_secret


AZURE_STORAGE_ACCOUNT = exigir_valor(
    get_secret(
        "AZURE-STORAGE-ACCOUNT",
        "AZURE_STORAGE_ACCOUNT",
    ),
    "AZURE_STORAGE_ACCOUNT",
)

AZURE_STORAGE_KEY = exigir_valor(
    get_secret(
        "AZURE-STORAGE-KEY",
        "AZURE_STORAGE_KEY",
    ),
    "AZURE_STORAGE_KEY",
)

BRONZE_CONTAINER = exigir_valor(
    get_secret(
        "AZURE-CONTAINER-BRONZE",
        "AZURE_CONTAINER_BRONZE",
    ),
    "AZURE_CONTAINER_BRONZE",
)

LOCAL_BRONZE_PATH = (
    BASE_DIR
    / "dados"
    / "bronze"
)

LOCAL_BRONZE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)


def validar_configuracao():
    if not ARQUIVOS:
        raise ValueError(
            "Nenhum arquivo foi configurado."
        )

    nomes = set()

    for indice, item in enumerate(
        ARQUIVOS,
        start=1,
    ):
        nome = str(
            item.get("nome", "")
        ).strip()

        url = str(
            item.get("url", "")
        ).strip()

        if not nome:
            raise ValueError(
                f"Entrada {indice} sem nome."
            )

        if not url:
            raise ValueError(
                f"Entrada {indice} sem URL."
            )

        if nome in nomes:
            raise ValueError(
                f"Nome duplicado: {nome}"
            )

        if not url.startswith(
            ("http://", "https://")
        ):
            raise ValueError(
                f"URL inválida para {nome}: "
                f"{url}"
            )

        nomes.add(nome)


def criar_sessao_http():
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        status=5,
        backoff_factor=2,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504,
        ],
        allowed_methods=frozenset([
            "GET",
        ]),
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry
    )

    session = requests.Session()

    session.mount(
        "https://",
        adapter,
    )

    session.mount(
        "http://",
        adapter,
    )

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 "
            "IBGE-Azure-Pipeline/1.0"
        ),
        "Accept": "application/json,*/*",
        "Connection": "close",
    })

    return session


def criar_container_client():
    credential = AzureNamedKeyCredential(
        name=AZURE_STORAGE_ACCOUNT,
        key=AZURE_STORAGE_KEY,
    )

    account_url = (
        f"https://{AZURE_STORAGE_ACCOUNT}"
        ".blob.core.windows.net"
    )

    blob_service_client = BlobServiceClient(
        account_url=account_url,
        credential=credential,
    )

    blob_service_client.get_account_information()

    container_client = (
        blob_service_client.get_container_client(
            container=BRONZE_CONTAINER
        )
    )

    container_client.get_container_properties()

    print(
        "✅ Cliente Azure Blob autenticado."
    )

    print(
        "✅ Ambiente:",
        "Databricks"
        if IS_DATABRICKS
        else "local",
    )

    print(
        f"✅ Storage Account: "
        f"{AZURE_STORAGE_ACCOUNT}"
    )

    print(
        f"✅ Contêiner Bronze: "
        f"{BRONZE_CONTAINER}"
    )

    print(
        f"✅ Pasta Bronze local: "
        f"{LOCAL_BRONZE_PATH.resolve()}"
    )

    return container_client


def baixar_arquivo(
    session,
    url,
    destino,
):
    destino = Path(destino)

    destino.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    arquivo_parcial = destino.with_suffix(
        destino.suffix + ".part"
    )

    if arquivo_parcial.exists():
        arquivo_parcial.unlink()

    try:
        with session.get(
            url=url,
            stream=True,
            verify=certifi.where(),
            timeout=(60, 3600),
            allow_redirects=True,
        ) as response:
            response.raise_for_status()

            content_type = (
                response.headers
                .get("Content-Type", "")
                .lower()
            )

            if "text/html" in content_type:
                raise RuntimeError(
                    "O servidor retornou HTML "
                    "em vez do arquivo esperado. "
                    f"URL final: {response.url}"
                )

            with arquivo_parcial.open(
                mode="wb"
            ) as arquivo:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        arquivo.write(chunk)

        if not arquivo_parcial.is_file():
            raise RuntimeError(
                "Arquivo parcial não foi criado."
            )

        tamanho = (
            arquivo_parcial.stat().st_size
        )

        if tamanho <= 0:
            raise RuntimeError(
                "O download produziu "
                "um arquivo vazio."
            )

        arquivo_parcial.replace(
            destino
        )

        print(
            f"✅ Download concluído: "
            f"{destino.name}"
        )

        print(
            f"✅ Arquivo local: "
            f"{destino.resolve()}"
        )

        print(
            f"✅ Tamanho local: "
            f"{tamanho / 1024:.2f} KB"
        )

        return destino

    except Exception:
        if arquivo_parcial.exists():
            arquivo_parcial.unlink()

        raise


def enviar_para_bronze(
    container_client,
    arquivo_local,
):
    arquivo_local = Path(
        arquivo_local
    )

    if not arquivo_local.is_file():
        raise FileNotFoundError(
            f"Arquivo local não encontrado: "
            f"{arquivo_local}"
        )

    tamanho_local = (
        arquivo_local.stat().st_size
    )

    if tamanho_local <= 0:
        raise RuntimeError(
            f"Arquivo local vazio: "
            f"{arquivo_local}"
        )

    blob_name = arquivo_local.name

    blob_client = (
        container_client.get_blob_client(
            blob=blob_name
        )
    )

    with arquivo_local.open(
        mode="rb"
    ) as arquivo:
        blob_client.upload_blob(
            data=arquivo,
            overwrite=OVERWRITE,
        )

    propriedades = (
        blob_client.get_blob_properties()
    )

    if propriedades.size != tamanho_local:
        raise RuntimeError(
            "O tamanho enviado nãocorresponde "
            "ao arquivo local. "
            f"Local: {tamanho_local} bytes. "
            f"Azure: {propriedades.size} bytes."
        )

    print(
        f"✅ Upload concluído: "
        f"{blob_name}"
    )

    print(
        f"✅ Tamanho no Azure: "
        f"{propriedades.size / 1024:.2f} KB"
    )

    return {
        "blob": blob_name,
        "arquivo_local": str(
            arquivo_local.resolve()
        ),
        "size_bytes": propriedades.size,
        "etag": str(
            propriedades.etag
        ),
        "last_modified": str(
            propriedades.last_modified
        ),
    }


def executar_pipeline():
    validar_configuracao()

    container_client = (
        criar_container_client()
    )

    session = criar_sessao_http()

    uploads_concluidos = []
    falhas = []

    try:
        total = len(ARQUIVOS)

        for indice, item in enumerate(
            ARQUIVOS,
            start=1,
        ):
            nome = item["nome"]
            url = item["url"]

            destino_local = (
                LOCAL_BRONZE_PATH
                / nome
            )

            print()
            print(
                f"[{indice}/{total}] "
                f"Processando: {nome}"
            )

            try:
                arquivo_baixado = baixar_arquivo(
                    session=session,
                    url=url,
                    destino=destino_local,
                )

                resultado = enviar_para_bronze(
                    container_client=container_client,
                    arquivo_local=arquivo_baixado,
                )

                uploads_concluidos.append({
                    "arquivo": nome,
                    "origem": url,
                    "container": BRONZE_CONTAINER,
                    **resultado,
                })

            except Exception as exc:
                erro = (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )

                falhas.append({
                    "arquivo": nome,
                    "origem": url,
                    "blob": nome,
                    "erro": erro,
                })

                print(
                    f"❌ Falha em {nome}: "
                    f"{erro}"
                )

    finally:
        session.close()

    print()
    print(
        "========== RESUMO =========="
    )

    print(
        f"✅ Arquivos enviados: "
        f"{len(uploads_concluidos)}"
    )

    print(
        f"❌ Falhas: "
        f"{len(falhas)}"
    )

    print(
        f"✅ Contêiner: "
        f"{BRONZE_CONTAINER}"
    )

    print(
        f"✅ Pasta local: "
        f"{LOCAL_BRONZE_PATH.resolve()}"
    )

    if falhas:
        detalhes = "\n".join(
            (
                f"- Arquivo: {falha['arquivo']}\n"
                f"  Origem: {falha['origem']}\n"
                f"  Blob: {falha['blob']}\n"
                f"  Erro: {falha['erro']}"
            )
            for falha in falhas
        )

        raise RuntimeError(
            f"{len(falhas)} arquivo(s) "
            "não foram enviados:\n"
            f"{detalhes}"
        )

    return (
        container_client,
        uploads_concluidos,
    )


def validar_bronze(
    container_client,
    uploads_concluidos,
):
    print()
    print(
        "========== VALIDAÇÃO BRONZE =========="
    )

    for upload in uploads_concluidos:
        arquivo_local = Path(
            upload["arquivo_local"]
        )

        blob_name = upload["blob"]

        if not arquivo_local.is_file():
            raise FileNotFoundError(
                f"Arquivo local não encontrado: "
                f"{arquivo_local}"
            )

        tamanho_local = (
            arquivo_local.stat().st_size
        )

        propriedades = (
            container_client
            .get_blob_client(
                blob=blob_name
            )
            .get_blob_properties()
        )

        if propriedades.size != tamanho_local:
            raise RuntimeError(
                f"Tamanho divergente em "
                f"{blob_name}. "
                f"Local: {tamanho_local}. "
                f"Azure: {propriedades.size}."
            )

        print(
            f"✅ Local: "
            f"{arquivo_local.name} "
            f"({tamanho_local / 1024:.2f} KB)"
        )

        print(
            f"✅ Azure: "
            f"{blob_name} "
            f"({propriedades.size / 1024:.2f} KB)"
        )

    blobs_enviados = {
        upload["blob"]
        for upload in uploads_concluidos
    }

    blobs_na_raiz = [
        blob
        for blob in container_client.list_blobs()
        if "/" not in blob.name
    ]

    print()

    print(
        f"✅ Contêiner validado: "
        f"{BRONZE_CONTAINER}"
    )

    print(
        f"✅ Arquivos desta execução: "
        f"{len(uploads_concluidos)}"
    )

    print(
        f"✅ Blobs na raiz do Bronze: "
        f"{len(blobs_na_raiz)}"
    )

    nao_encontrados = (
        blobs_enviados
        - {
            blob.name
            for blob in blobs_na_raiz
        }
    )

    if nao_encontrados:
        raise RuntimeError(
            "Blobs não encontrados na raiz "
            f"do contêiner: "
            f"{sorted(nao_encontrados)}"
        )

    print(
        "✅ Arquivos locais e blobs "
        "validados com sucesso."
    )


(
    container_client,
    uploads_concluidos,
) = executar_pipeline()

validar_bronze(
    container_client=container_client,
    uploads_concluidos=uploads_concluidos,
)

✅ Cliente Azure Blob autenticado.
✅ Ambiente: local
✅ Storage Account: stfiaptechchallenge2iast
✅ Contêiner Bronze: bronze
✅ Pasta Bronze local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze

[1/2] Processando: pib_municipal.json
✅ Download concluído: pib_municipal.json
✅ Arquivo local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/pib_municipal.json
✅ Tamanho local: 10191.82 KB
✅ Upload concluído: pib_municipal.json
✅ Tamanho no Azure: 10191.82 KB

[2/2] Processando: censo_2022.json
✅ Download concluído: censo_2022.json
✅ Arquivo local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/censo_2022.json
✅ Tamanho local: 4542.91 KB
✅ Upload concluído: censo_2022.json
✅ Tamanho no Azure: 4542.91 KB

========== RESUMO ==========
✅ Arquivos enviados: 2
❌ Falhas: 0
✅ Contêiner: bronze
✅ Pasta local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze

========== VALIDAÇÃO BRONZE ========

In [4]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import json
import re

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print


# ============================================================
# CONFIGURAÇÕES
# ============================================================

FUSO_HORARIO = ZoneInfo(
    "America/Sao_Paulo"
)

DATA_EXECUCAO = datetime.now(
    FUSO_HORARIO
).strftime("%Y-%m-%d")

ARQUIVOS_ORIGINAIS = {
    "PIB municipal": "pib_municipal.json",
    "Censo 2022": "censo_2022.json",
}


# ============================================================
# LOCALIZAR PASTA BRONZE
# ============================================================

def localizar_pasta_bronze() -> Path:
    diretorio_atual = Path.cwd().resolve()

    for diretorio in [
        diretorio_atual,
        *diretorio_atual.parents,
    ]:
        pasta_bronze = (
            diretorio
            / "dados"
            / "bronze"
        )

        if pasta_bronze.is_dir():
            return pasta_bronze

    raise FileNotFoundError(
        "A pasta dados/bronze não foi encontrada "
        "no diretório atual nem nos diretórios superiores."
    )


# ============================================================
# GERAR NOME COM PREFIXO YYYY-MM-DD
# ============================================================

def gerar_nome_datado(
    nome_original: str,
) -> str:
    padrao_data = re.compile(
        r"^\d{4}-\d{2}-\d{2}_"
    )

    # Evita acrescentar uma segunda data.
    if padrao_data.match(nome_original):
        return nome_original

    return (
        f"{DATA_EXECUCAO}_"
        f"{nome_original}"
    )


# ============================================================
# LOCALIZAR ARQUIVO DATADO MAIS RECENTE
# ============================================================

def localizar_arquivo_datado_mais_recente(
    pasta: Path,
    nome_original: str,
) -> Path | None:
    caminho_nome = Path(
        nome_original
    )

    stem = caminho_nome.stem
    suffix = caminho_nome.suffix

    padrao_nome = re.compile(
        rf"^\d{{4}}-\d{{2}}-\d{{2}}_"
        rf"{re.escape(stem)}"
        rf"{re.escape(suffix)}$"
    )

    candidatos = [
        arquivo
        for arquivo in pasta.glob(
            f"*_{stem}{suffix}"
        )
        if (
            arquivo.is_file()
            and padrao_nome.match(
                arquivo.name
            )
        )
    ]

    if not candidatos:
        return None

    # O formato YYYY-MM-DD pode ser ordenado
    # corretamente pelo próprio nome.
    return max(
        candidatos,
        key=lambda arquivo: arquivo.name,
    )


# ============================================================
# RENOMEAR JSON SEM ALTERAR O CONTEÚDO
# ============================================================

def renomear_json_com_data(
    pasta: Path,
    nome_original: str,
) -> Path:
    arquivo_original = (
        pasta
        / nome_original
    )

    nome_datado = gerar_nome_datado(
        nome_original
    )

    arquivo_datado = (
        pasta
        / nome_datado
    )

    # Caso o arquivo desta data já exista,
    # usa o arquivo existente sem alterá-lo.
    if arquivo_datado.is_file():
        print(
            f"ℹ️ Arquivo datado já existe: "
            f"{arquivo_datado.name}"
        )

        return arquivo_datado

    # Renomeia somente o arquivo.
    # O conteúdo interno do JSON não é modificado.
    if arquivo_original.is_file():
        arquivo_original.rename(
            arquivo_datado
        )

        print(
            f"✅ Arquivo renomeado: "
            f"{nome_original}"
        )

        print(
            f"✅ Novo nome: "
            f"{arquivo_datado.name}"
        )

        return arquivo_datado

    # Se o original já foi renomeado anteriormente,
    # utiliza o arquivo datado mais recente.
    arquivo_mais_recente = (
        localizar_arquivo_datado_mais_recente(
            pasta=pasta,
            nome_original=nome_original,
        )
    )

    if arquivo_mais_recente is not None:
        print(
            f"ℹ️ Arquivo original não encontrado. "
            f"Utilizando o datado mais recente: "
            f"{arquivo_mais_recente.name}"
        )

        return arquivo_mais_recente

    raise FileNotFoundError(
        f"Não foi encontrado o arquivo original "
        f"nem uma versão datada de: "
        f"{nome_original}"
    )


# ============================================================
# IDENTIFICAR CABEÇALHO SIDRA
# ============================================================

def identificar_cabecalho_sidra(
    primeiro_registro: dict,
) -> bool:
    if not primeiro_registro:
        return False

    if not all(
        isinstance(valor, str)
        for valor in primeiro_registro.values()
    ):
        return False

    termos_cabecalho = [
        "município",
        "municipio",
        "variável",
        "variavel",
        "valor",
        "unidade",
        "ano",
        "nível territorial",
        "nivel territorial",
        "unidade da federação",
        "unidade da federacao",
    ]

    valores_normalizados = [
        str(valor)
        .strip()
        .lower()
        for valor in primeiro_registro.values()
    ]

    return any(
        termo in valor
        for valor in valores_normalizados
        for termo in termos_cabecalho
    )


# ============================================================
# CARREGAR JSON DO SIDRA
# ============================================================

def carregar_json_sidra(
    caminho: Path,
) -> pd.DataFrame:
    caminho = Path(
        caminho
    )

    if not caminho.is_file():
        raise FileNotFoundError(
            f"Arquivo não encontrado: "
            f"{caminho}"
        )

    # O modo "r" abre o arquivo somente para leitura.
    # Nenhum conteúdo será modificado.
    with caminho.open(
        mode="r",
        encoding="utf-8-sig",
    ) as arquivo:
        dados = json.load(
            arquivo
        )

    if not isinstance(
        dados,
        list,
    ):
        raise TypeError(
            f"O arquivo {caminho.name} não contém "
            "uma lista JSON."
        )

    if not dados:
        raise ValueError(
            f"O arquivo {caminho.name} está vazio."
        )

    registros_invalidos = [
        indice
        for indice, registro in enumerate(
            dados
        )
        if not isinstance(
            registro,
            dict,
        )
    ]

    if registros_invalidos:
        raise TypeError(
            f"O arquivo {caminho.name} contém "
            "registros em formato inesperado. "
            f"Primeiros índices inválidos: "
            f"{registros_invalidos[:10]}"
        )

    primeiro_registro = dados[0]

    parece_cabecalho_sidra = (
        identificar_cabecalho_sidra(
            primeiro_registro
        )
    )

    if parece_cabecalho_sidra:
        dataframe = pd.DataFrame(
            dados[1:]
        ).rename(
            columns=primeiro_registro
        )

    else:
        dataframe = pd.json_normalize(
            dados,
            sep="_",
        )

    dataframe.columns = [
        str(coluna).strip()
        for coluna in dataframe.columns
    ]

    return dataframe


# ============================================================
# LOCALIZAR E RENOMEAR OS ARQUIVOS
# ============================================================

PASTA_BRONZE = localizar_pasta_bronze()

arquivos = {}
falhas_renomeacao = []

print(
    f"✅ Pasta Bronze: "
    f"{PASTA_BRONZE.resolve()}"
)

print(
    f"✅ Data utilizada no prefixo: "
    f"{DATA_EXECUCAO}"
)

for titulo, nome_original in (
    ARQUIVOS_ORIGINAIS.items()
):
    try:
        arquivo_datado = renomear_json_com_data(
            pasta=PASTA_BRONZE,
            nome_original=nome_original,
        )

        arquivos[
            titulo
        ] = arquivo_datado

    except Exception as erro:
        falhas_renomeacao.append({
            "titulo": titulo,
            "arquivo": nome_original,
            "erro": (
                f"{type(erro).__name__}: "
                f"{erro}"
            ),
        })

        print(
            f"❌ Erro ao renomear "
            f"{nome_original}: "
            f"{type(erro).__name__}: "
            f"{erro}"
        )


# ============================================================
# CARREGAR E MOSTRAR OS ARQUIVOS
# ============================================================

dataframes = {}
falhas_processamento = []

for titulo, caminho in arquivos.items():
    print()
    print(
        "=" * 100
    )

    print(
        titulo
    )

    print(
        "=" * 100
    )

    try:
        dataframe = carregar_json_sidra(
            caminho
        )

        dataframes[
            titulo
        ] = dataframe

        tamanho_bytes = (
            caminho.stat().st_size
        )

        print(
            f"✅ Arquivo: "
            f"{caminho.name}"
        )

        print(
            f"✅ Caminho completo: "
            f"{caminho.resolve()}"
        )

        print(
            f"✅ Tamanho: "
            f"{tamanho_bytes / 1024:.2f} KB"
        )

        print(
            f"✅ Registros: "
            f"{len(dataframe):,}"
        )

        print(
            f"✅ Colunas: "
            f"{len(dataframe.columns)}"
        )

        if dataframe.empty:
            print(
                "⚠️ O DataFrame está vazio."
            )

            continue

        print()
        print(
            "✅ Nomes das colunas:"
        )

        for indice, coluna in enumerate(
            dataframe.columns,
            start=1,
        ):
            print(
                f"   {indice}. {coluna}"
            )

        print()
        print(
            "✅ Primeiras 10 linhas:"
        )

        display(
            dataframe.head(10)
        )

    except Exception as erro:
        falhas_processamento.append({
            "titulo": titulo,
            "arquivo": caminho.name,
            "erro": (
                f"{type(erro).__name__}: "
                f"{erro}"
            ),
        })

        print(
            f"❌ Erro ao processar "
            f"{caminho.name}: "
            f"{type(erro).__name__}: "
            f"{erro}"
        )


# ============================================================
# RESUMO
# ============================================================

print()
print(
    "=" * 100
)

print(
    "RESUMO"
)

print(
    "=" * 100
)

print(
    f"✅ Arquivos renomeados ou localizados: "
    f"{len(arquivos)}"
)

print(
    f"✅ Arquivos carregados: "
    f"{len(dataframes)}"
)

print(
    f"❌ Falhas de renomeação: "
    f"{len(falhas_renomeacao)}"
)

print(
    f"❌ Falhas de processamento: "
    f"{len(falhas_processamento)}"
)


if falhas_renomeacao:
    print()
    print(
        "Falhas de renomeação:"
    )

    for falha in falhas_renomeacao:
        print(
            f"   • {falha['titulo']} "
            f"({falha['arquivo']}): "
            f"{falha['erro']}"
        )


if falhas_processamento:
    print()
    print(
        "Falhas de processamento:"
    )

    for falha in falhas_processamento:
        print(
            f"   • {falha['titulo']} "
            f"({falha['arquivo']}): "
            f"{falha['erro']}"
        )


# ============================================================
# DATAFRAMES RESULTANTES
# ============================================================

df_pib = dataframes.get(
    "PIB municipal"
)

df_censo = dataframes.get(
    "Censo 2022"
)


print()
print(
    "✅ Execução concluída."
)

print(
    "ℹ️ Somente os nomes dos arquivos JSON "
    "foram alterados."
)

print(
    "ℹ️ O conteúdo interno dos arquivos JSON "
    "permaneceu intacto."
)

✅ Pasta Bronze: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze
✅ Data utilizada no prefixo: 2026-10-18
✅ Arquivo renomeado: pib_municipal.json
✅ Novo nome: 2026-10-18_pib_municipal.json
✅ Arquivo renomeado: censo_2022.json
✅ Novo nome: 2026-10-18_censo_2022.json

PIB municipal
✅ Arquivo: 2026-10-18_pib_municipal.json
✅ Caminho completo: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/2026-10-18_pib_municipal.json
✅ Tamanho: 10191.82 KB
✅ Registros: 27,850
✅ Colunas: 11

✅ Nomes das colunas:
   1. Nível Territorial (Código)
   2. Nível Territorial
   3. Unidade de Medida (Código)
   4. Unidade de Medida
   5. Valor
   6. Município (Código)
   7. Município
   8. Variável (Código)
   9. Variável
   10. Ano (Código)
   11. Ano

✅ Primeiras 10 linhas:


,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano
0,6,Município,40,Mil Reais,734467,1100015,Alta Floresta D'Oeste - RO,37,Produto Interno Bruto a preços correntes,2021,2021
1,6,Município,2,%,45.48,1100015,Alta Floresta D'Oeste - RO,516,Participação do valor adicionado bruto a preço...,2021,2021
2,6,Município,2,%,4.07,1100015,Alta Floresta D'Oeste - RO,520,Participação do valor adicionado bruto a preço...,2021,2021
3,6,Município,2,%,25.28,1100015,Alta Floresta D'Oeste - RO,528,Participação do valor adicionado bruto a preço...,2021,2021
4,6,Município,2,%,25.18,1100015,Alta Floresta D'Oeste - RO,6574,Participação do valor adicionado bruto a preço...,2021,2021
5,6,Município,40,Mil Reais,3211294,1100023,Ariquemes - RO,37,Produto Interno Bruto a preços correntes,2021,2021
6,6,Município,2,%,10.50,1100023,Ariquemes - RO,516,Participação do valor adicionado bruto a preço...,2021,2021
7,6,Município,2,%,14.61,1100023,Ariquemes - RO,520,Participação do valor adicionado bruto a preço...,2021,2021
8,6,Município,2,%,28.03,1100023,Ariquemes - RO,528,Participação do valor adicionado bruto a preço...,2021,2021
9,6,Município,2,%,46.86,1100023,Ariquemes - RO,6574,Participação do valor adicionado bruto a preço...,2021,2021



Censo 2022
✅ Arquivo: 2026-10-18_censo_2022.json
✅ Caminho completo: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/2026-10-18_censo_2022.json
✅ Tamanho: 4542.91 KB
✅ Registros: 16,710
✅ Colunas: 11

✅ Nomes das colunas:
   1. Nível Territorial (Código)
   2. Nível Territorial
   3. Unidade de Medida (Código)
   4. Unidade de Medida
   5. Valor
   6. Município (Código)
   7. Município
   8. Variável (Código)
   9. Variável
   10. Ano (Código)
   11. Ano

✅ Primeiras 10 linhas:


,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano
0,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022
1,6,Município,26,Quilômetros quadrados,7067.127,1100015,Alta Floresta D'Oeste - RO,6318,Área da unidade territorial,2022,2022
2,6,Município,28,Habitante por quilômetro quadrado,3.04,1100015,Alta Floresta D'Oeste - RO,614,Densidade demográfica,2022,2022
3,6,Município,45,Pessoas,96833,1100023,Ariquemes - RO,93,População residente,2022,2022
4,6,Município,26,Quilômetros quadrados,4426.571,1100023,Ariquemes - RO,6318,Área da unidade territorial,2022,2022
5,6,Município,28,Habitante por quilômetro quadrado,21.88,1100023,Ariquemes - RO,614,Densidade demográfica,2022,2022
6,6,Município,45,Pessoas,5351,1100031,Cabixi - RO,93,População residente,2022,2022
7,6,Município,26,Quilômetros quadrados,1314.352,1100031,Cabixi - RO,6318,Área da unidade territorial,2022,2022
8,6,Município,28,Habitante por quilômetro quadrado,4.07,1100031,Cabixi - RO,614,Densidade demográfica,2022,2022
9,6,Município,45,Pessoas,86887,1100049,Cacoal - RO,93,População residente,2022,2022



RESUMO
✅ Arquivos renomeados ou localizados: 2
✅ Arquivos carregados: 2
❌ Falhas de renomeação: 0
❌ Falhas de processamento: 0

✅ Execução concluída.
ℹ️ Somente os nomes dos arquivos JSON foram alterados.
ℹ️ O conteúdo interno dos arquivos JSON permaneceu intacto.
